# 04 Data Preprocessing

This notebook cleans the messy synthetic copy while preserving the original input files. The final grain is one row per order.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Run this notebook from inside the EcomPolice project.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CLEAN_DIR = PROJECT_ROOT / "data" / "synthetic" / "clean"
MESSY_DIR = PROJECT_ROOT / "data" / "synthetic" / "messy"
LABELS_DIR = PROJECT_ROOT / "data" / "synthetic" / "labels"
MANIFESTS_DIR = PROJECT_ROOT / "data" / "synthetic" / "manifests"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def load_tables(base_dir: Path) -> dict[str, pd.DataFrame]:
    return {
        "customers": pd.read_csv(base_dir / "customers.csv"),
        "products": pd.read_csv(base_dir / "products.csv"),
        "orders": pd.read_csv(base_dir / "orders.csv"),
        "deliveries": pd.read_csv(base_dir / "deliveries.csv"),
        "returns": pd.read_csv(base_dir / "returns.csv"),
    }

from synthetic_data.preprocessing import SyntheticTables, build_clean_order_dataset, preprocess_messy_tables, quality_summary
messy = load_tables(MESSY_DIR)
manifest = pd.read_csv(MANIFESTS_DIR / 'defect_manifest.csv')
raw_tables = SyntheticTables(**messy)


In [2]:
before_orders = messy['orders'].copy()
print('Original messy order rows:', len(before_orders))
display(manifest['Defect_Type'].value_counts().rename_axis('defect_type').reset_index(name='count'))


Original messy order rows: 20060


,defect_type,count
0,mixed_date_format,300
1,inconsistent_category,120
2,numeric_as_string,120
3,missing_value,80
4,duplicate_row,60
5,invalid_date_chronology,60
6,missing_identifier,40
7,impossible_negative_value,40
8,implausible_outlier,40


In [3]:
cleaned_tables, action_log = preprocess_messy_tables(raw_tables)
cleaned_orders = build_clean_order_dataset(cleaned_tables)
display(action_log)
display(quality_summary(cleaned_orders).to_frame('after_cleaning'))


,step,action,affected_rows
0,categorical_normalization,Corrected unambiguous payment and shipping spe...,NaN
1,duplicate_rows,Dropped exact duplicate order rows only.,58.0
2,parse_Order_Date,Parsed ISO and day-first date formats.,0.0
3,parse_Dispatch_Date,Parsed delivery date values.,0.0
4,parse_Delivery_Date,Parsed delivery date values.,0.0
5,parse_Quantity,Converted numeric-looking strings to numeric v...,0.0
6,parse_Unit_Price_INR,Converted numeric-looking strings to numeric v...,0.0
7,parse_Discount_Pct,Converted numeric-looking strings to numeric v...,0.0
8,parse_Order_Value_INR,Converted numeric-looking strings to numeric v...,0.0
9,quality_flags,"Flagged invalid identifiers, numeric defects, ...",NaN


,after_cleaning
rows,20000
unique_orders,20000
duplicate_order_ids,0
invalid_identifiers,40
invalid_numeric_rows,39
suspicious_outlier_rows,23
invalid_return_chronology,58
missing_customer_joins,20
missing_product_joins,20


In [4]:
before_summary = pd.Series({
    'rows': len(before_orders),
    'unique_orders': before_orders['Order_ID'].nunique(dropna=True),
    'duplicate_order_ids': int(before_orders['Order_ID'].duplicated().sum()),
    'missing_customer_ids': int(before_orders['Customer_ID'].isna().sum() + before_orders['Customer_ID'].astype(str).str.strip().eq('').sum()),
    'missing_product_ids': int(before_orders['Product_ID'].isna().sum() + before_orders['Product_ID'].astype(str).str.strip().eq('').sum()),
})
after_summary = quality_summary(cleaned_orders)
comparison = pd.concat([before_summary.rename('before_preprocessing'), after_summary.rename('after_preprocessing')], axis=1)
display(comparison)


,before_preprocessing,after_preprocessing
rows,20060.0,20000.0
unique_orders,20000.0,20000.0
duplicate_order_ids,60.0,0.0
missing_customer_ids,20.0,NaN
missing_product_ids,20.0,NaN
invalid_identifiers,NaN,40.0
invalid_numeric_rows,NaN,39.0
suspicious_outlier_rows,NaN,23.0
invalid_return_chronology,NaN,58.0
missing_customer_joins,NaN,20.0


In [5]:
invalid_rows = cleaned_orders[
    cleaned_orders.get('Invalid_Identifier_Flag', False)
    | cleaned_orders.get('Invalid_Numeric_Flag', False)
    | cleaned_orders.get('Invalid_Return_Chronology_Flag', False)
    | cleaned_orders.get('Suspicious_Outlier_Flag', False)
]
display(invalid_rows[['Order_ID','Customer_ID','Product_ID','Invalid_Identifier_Flag','Invalid_Numeric_Flag','Invalid_Return_Chronology_Flag','Suspicious_Outlier_Flag']].head(10))


,Order_ID,Customer_ID,Product_ID,Invalid_Identifier_Flag,Invalid_Numeric_Flag,Invalid_Return_Chronology_Flag,Suspicious_Outlier_Flag
288,ORD0016987,CUST000352,PROD00062,False,True,False,False
307,ORD0004413,CUST001592,PROD00051,False,False,False,True
637,ORD0003524,CUST001048,PROD00195,False,False,False,True
654,ORD0009747,CUST001545,PROD00478,False,False,True,False
783,ORD0006362,CUST000036,PROD00048,False,True,False,False
815,ORD0017139,CUST000307,PROD00054,False,False,True,False
874,ORD0002334,CUST000215,NaN,True,False,False,False
940,ORD0008170,CUST000440,PROD00233,False,True,False,False
1030,ORD0000209,CUST001134,PROD00281,False,False,True,False
1254,ORD0018749,CUST000217,PROD00061,False,True,False,False


In [6]:
cleaned_orders.to_csv(PROCESSED_DIR / 'cleaned_orders.csv', index=False)
report = [
    '# Preprocessing Report',
    '## Strategy',
    '- Original messy files were loaded from `data/synthetic/messy/` and were not modified in place.',
    '- Column names were standardized, mixed dates and numeric-looking strings were parsed, and unambiguous category variants were normalized.',
    '- Exact duplicate order rows were removed, while invalid identifiers, invalid numeric values, chronology issues, and outliers were retained with quality flags where possible.',
    '- The consolidated dataset uses one row per order and avoids accidental row multiplication from joins.',
    '## Quality Metrics',
    '```text\n' + comparison.to_string() + '\n```',
    '## Known Limits',
    '- When an injected defect destroys the original value, this notebook flags the issue instead of claiming perfect recovery.',
]
(REPORTS_DIR / 'preprocessing_report.md').write_text('\n\n'.join(report), encoding='utf-8')
print('Saved:', PROCESSED_DIR / 'cleaned_orders.csv')


Saved: /Users/harshitmadhavpachauri/Documents/ChatGPT/EcomPolice/data/processed/cleaned_orders.csv


## Summary

The cleaned analytical dataset preserves one row per order. Quality flags make questionable records visible for future decisions instead of silently deleting all unusual values.
